In [28]:
import numpy as np
def build_causal_mask(seq_len):
    upper=np.triu(np.ones((seq_len,seq_len),dtype=np.int32),k=1)
    return upper==0

def matmul(a,b):
    return np.einsum('ij,jk->ik',a,b)
a=np.random.rand(3,2)*10
b=np.random.rand(2,3)
def batch_attention_scores(q,k):
    return np.einsum('bqd,bkd->bqk',q,k)
print(a,b)
print(matmul(a,b))
                                      
print(build_causal_mask(5))

[[2.35898607 2.75088172]
 [0.68099243 7.44790363]
 [3.93761167 1.88223218]] [[0.24947142 0.57317569 0.33048653]
 [0.84737319 0.13810492 0.87523619]]
[[2.91952304 1.73202375 3.18728436]
 [6.48104204 1.41892042 6.74373363]
 [2.57727468 2.51688879 2.94872534]]
[[ True False False False False]
 [ True  True False False False]
 [ True  True  True False False]
 [ True  True  True  True False]
 [ True  True  True  True  True]]


In [2]:

import torch
x=torch.tensor([1,2,3])
print(x)


tensor([1, 2, 3])


tensor([[[-1.4492e+00,  1.0131e+00, -1.4440e-01,  9.6917e-01],
         [ 2.0633e+00, -3.3713e-04, -2.0967e-01,  1.2140e+00],
         [ 1.1136e+00, -5.4125e-01, -7.8961e-01,  7.0979e-01]],

        [[-4.0410e-01,  7.0058e-01, -7.3687e-01,  1.3354e+00],
         [-2.0918e+00,  8.6710e-01, -1.7340e+00, -8.9342e-01],
         [-1.4953e+00, -5.0927e-03, -3.6197e-01,  1.6635e+00]]])
tensor([-1.4492e+00,  1.0131e+00, -1.4440e-01,  9.6917e-01,  2.0633e+00,
        -3.3713e-04, -2.0967e-01,  1.2140e+00,  1.1136e+00, -5.4125e-01,
        -7.8961e-01,  7.0979e-01, -4.0410e-01,  7.0058e-01, -7.3687e-01,
         1.3354e+00, -2.0918e+00,  8.6710e-01, -1.7340e+00, -8.9342e-01,
        -1.4953e+00, -5.0927e-03, -3.6197e-01,  1.6635e+00])


In [35]:
def create_causal_mask(seq_len):

    inp=torch.randn(seq_len,seq_len)
    mask=torch.triu(inp,diagonal=1)
    mask=mask==0
    return mask

mask = create_causal_mask(4)
print(mask)

tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])


In [33]:
def split_heads(x, num_heads):
   
    batch, seq, hidden = x.shape
    head_dim = hidden // num_heads
    x=torch.reshape(x,(batch,seq,num_heads,head_dim))
    x=torch.permute(x,(0,2,1,3))

    return x


x = torch.randn(2, 10, 512)
x_split = split_heads(x, num_heads=8)
print(x_split.shape)  # torch.Size([2, 8, 10, 64])

torch.Size([2, 8, 10, 64])


In [36]:
def batch_matmul(a, b):
 
    res=torch.bmm(a,b)
    # return res
    return a@b

a = torch.randn(2, 3, 4)
b = torch.randn(2, 4, 5)
c = batch_matmul(a, b)
print(c.shape)  # torch.Size([2, 3, 5])

torch.Size([2, 3, 5])


In [38]:
import torch
x=torch.tensor([2.0],requires_grad=True)
y=x**2
y.backward()
print(x.grad)

tensor([4.])


In [42]:
x = torch.tensor([2.0], requires_grad=True)
y = torch.tensor([3.0], requires_grad=True)

# z = x^2 + y^2
z = x ** 2 + y ** 2

# 反向传播
z.backward()
y.grad.zero_()
o=x**2+y**2
o.backward()
# 查看梯度
print(x.grad)  # tensor([4.])  因为 dz/dx = 2x = 4
print(y.grad)  # tensor([6.])  因为 dz/dy = 2y = 6

tensor([8.])
tensor([6.])


In [43]:
x = torch.tensor([2.0], requires_grad=True)

y = x ** 2
print(y.requires_grad)  # True

with torch.no_grad():
    y = x ** 2
    print(y.requires_grad)  # False

model.eval()
with torch.no_grad():
    output = model(input)

True
False


NameError: name 'model' is not defined

In [44]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
#detach removes the var from the comp graph
y_detached = y.detach()
print(y_detached.requires_grad)  # False

z = y_detached * 3
z.backward()  #givees error here 

False


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
#we can als90 define funcs as non grad and avoid usingv the wiht torch.nograda
@torch.no_grad()
def inference(model, input):
    return model(input)

In [7]:
import torch
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2

# 第一次反向传播
y.backward(retain_graph=True)
print(x.grad)  # tensor([4.])

# 第二次反向传播（需要 retain_graph=True）
x.grad.zero_()
y.backward()
print(x.grad)  # tensor([4.])

tensor([4.])
tensor([4.])


In [9]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x ** 2  # y = [4, 9]

# 指定 dy = [1, 2]
y.backward(torch.tensor([1.0, 2.0]))

# dx = dy * dy/dx = [1, 2] * [4, 6] = [4, 12]
print(x.grad)  # tensor([4., 12.])

tensor([ 4., 12.])


In [ ]:
class MySoftmax(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        """
        Softmax 前向传播
        
        Args:
            input: 形状为 (batch, num_classes) 的 Tensor
        
        Returns:
            output: Softmax 输出
        """
        # TODO: 实现 Softmax
        # 提示: softmax(x) = exp(x) / sum(exp(x))
        # 注意数值稳定性: softmax(x) = exp(x - max(x)) / sum(exp(x - max(x)))
        
        ctx.save_for_backward(input)
        exp_x=torch.exp(input-torch.max(input[:-1]))
        sum_x=torch.sum(exp_x)
        return exp_x/sum_x
    
    @staticmethod
    def backward(ctx, grad_output):
        """
        Softmax 反向传播
        
        Args:
            grad_output: 输出的梯度
        
        Returns:
            grad_input: 输入的梯度
        """
        # TODO: 实现 Softmax 的梯度
        # 提示: ∂softmax_i/∂x_j = softmax_i * (δ_ij - softmax_j)
        outs,_=ctx.saved_tensors
        out=out*grad_output
        grad_input=
        pass

# 测试
softmax = MySoftmax.apply
x = torch.randn(2, 3, requires_grad=True)
y = softmax(x)
print(y)
print(y.sum(dim=1))  # 应该全为 1

In [1]:
def train_with_gradient_accumulation(model, data_loader, optimizer, accumulation_steps=4):
    """
    使用梯度累积模拟大 batch size
    
    Args:
        model: 模型
        data_loader: 数据加载器
        optimizer: 优化器
        accumulation_steps: 累积步数
    """
    model.train()
    optimizer.zero_grad()
    
    for i, (inputs, targets) in enumerate(data_loader):
        # TODO: 实现梯度累积
        # 提示:
        # 1. 前向传播
        # 2. 计算损失（除以 accumulation_steps）
        # 3. 反向传播
        # 4. 每 accumulation_steps 步更新一次参数
        outputs=model(inputs)
        loss=criterion(outputs,targets)
        loss=loss/accumulation_steps
        loss.backward()
        if (i+1)%accumulation_steps==0:
            optimizer.step()
            optimizer.zero_grad()
            

# 测试
# model = ...
# data_loader = ...
# optimizer = ...
# train_with_gradient_accumulation(model, data_loader, optimizer, accumulation_steps=4)

In [ ]:
def clip_grads(model,max_norm=1.0):
    totnorm=0.0
    for p in model.parameters():
        if p.grad is not None:
            parm_norm+=p.grad.data.norm(2)
            totnorm+=parm_norm**2
    totnorm=totnorm**0.5
    clip_coef=max_norm/(totnorm+1e-6)
    if clip_coef<1:
        for p in model.parameters():
            if p.grad is not None:
                p.grad.data.mul_(clip_coef)
    return totnorm]
# equivalent is 
#torch.nn,utils.clip_grad_norm_(model.parameters(),max_norm=1.0)


In [1]:
import torch.nn as nn
import torch
class Layer1(nn.Module):
    def __init__(self,fanin,fanout):
        super().__init__()
        self.w=nn.Parameter(torch.randn(fanin,fanout))
        self.b=nn.Parameter(torch.zeros(fanout))
        nn.init.xavier_uniform_(self.w)
    def forward(self,x):
        return x@self.w +self.b

class TwoLayer(nn.Module):
    def __init__(self,fanin,hidden,fanout):
        super().__init__()
        self.fc1=Layer1(fanin,hidden)
        self.act=nn.GELU(l1)
        self.fc2=Layer1(hidden,fanout)
    def forward(self,x):
        return self.fc2(self.act(self.fc1(x)))
def count_params(model):
    return sum(param.numel() for param in module.parameters())
    
    

In [9]:
def mse_loss(output,target):
    return torch.mean((output-target)**2)
def cross_entropy(out,tar):
    return torch.nn.cross_entropy(out,tar)
def model_train(model,x,tar,optimiser):
    model.train()
    optimiser.zero_grad()
    out=model(x)
    loss=mse_loss(out,tar)
    loss.backward()
    optimiser.step()
    return loss.item()

model = nn.Linear(1, 1)
with torch.no_grad():
    model.weight.zero_()
    model.bias.zero_()

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
target = 2 * x + 1
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

print('loss:', mse_loss(model(x), target).item())
for step in range(3):
    loss =model_train(model, x, target, optimizer)
    print(f'step {step + 1}: loss={loss:.4f}')
print('output:', model(x).detach().squeeze().tolist())

    


loss: 41.0
step 1: loss=41.0000
step 2: loss=1.1288
step 3: loss=0.0433
output: [2.8066248893737793, 4.890375137329102, 6.974125385284424, 9.05787467956543]


In [ ]:
#the basic structure of a training loop is this 
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        optimiser.zero_grad()
        out=model(input)
        loss=criterion(out,target)
        loss.backward()
        optimiser.step()
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            out=model(input)
            val_loss=criterion(out,tar)
            
# imma end this one here next we will cover the basics of training techniques and stuff like that all the way upto attention then we move to cuda 